# 第9章 文脈を広げる：埋め込みと多層パーセプトロン

直前8文字を見る、本物のニューラル言語モデル

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：直前8文字から次の文字を予測するニューラル言語モデル（MLP）。初めて「数えるbigram」の記録を破る
- **身につく数学・理論**：埋め込み（embedding）：文字を数の組（ベクトル）で表す ／ ニューロンと全結合層 $y = xW + b$ ／ 活性化関数（tanh・ReLU・GELU）となぜ非線形が必要か ／ パラメータ数の数え方 ／ 汎化：見たことのない並びにも対応できる理由
- **新しく使うPython・PyTorch**：`class` と `__init__`、`self` ／ `nn.Module` と `forward` ／ `nn.Embedding` と `nn.Linear` ／ `torch.optim.SGD`（`zero_grad` と `step`） ／ `.view` で形を変える

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 90〜120分 |
| 難易度 | ★★★☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第4章で「直前の文字をたくさん見ると、組み合わせが爆発して数えられない」という壁にぶつかりました。この章では、その壁を**埋め込み**とニューラルネットワークで越えます。直前の**8文字**を見て次の1文字を予測するモデルを作り、ついに「数えるbigram」の記録を破ります。2003年に発表された、ニューラル言語モデルの元祖とも言える設計です。

> **🔗 これまでの復習とつながり：第4章・第6章・第8章とのつながり**
>
> - **第4章（n-gramの組み合わせ爆発）**：直前8文字の並びをそのまま表にすると $2061^8$ 通り以上になり、数える方法では手も足も出ませんでした。
> - **第6章（行列積 `@` と one-hot × 表）**：行列の掛け算 `@` で複数のニューロンを一括計算できること、one-hot × 表は「表の行の取り出し」と同じであることを学びました。
> - **第8章（ニューラルbigramと限界）**：`F.cross_entropy` とミニバッチSGDで学習できることを確かめましたが、直前1文字だけの構造では検証損失 `3.581` の壁を越えられませんでした。この章でいよいよその壁を破ります！

**ミナト**：8文字も見るなら、組み合わせは $2061^8$ 通り……？ 数える気が失せる数ですね。

**ボンゴ先生**：だから数えない。代わりに、文字の「**特徴**」を学ばせる。「先」と「奥」はどちらも人を呼ぶ言葉の前に来やすい、とか、「。」の次は文の始まりだ、とかね。特徴でとらえれば、見たことのない並びでも「似たような並びは見たことがある」と推測できる。人間の読書と同じだ。

## 準備

いつもの準備セルです（第8章と同じ）。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        with urllib.request.urlopen(url, timeout=30) as res:
            text = res.read().decode("cp932")
    except Exception as e:
        raise RuntimeError(f"青空文庫の取得に失敗しました（時間をおいて再実行するか付録Eを参照）: {e}") from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

## 8文字の窓

文章の上に「8文字分の窓」を置いて、窓の中の8文字を入力、その次の1文字を正解にします。窓を1文字ずつずらせば、問題がたくさん作れます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
T = 8   # 窓の大きさ（何文字前まで見るか）

def get_batch(data, batch_size, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)  # 窓の開始位置をランダムに
    x = torch.stack([data[i:i + T] for i in ix])     # 入力：8文字（stack は、同じ形のテンソルを積み重ねて1つにする）
    y = data[ix + T]                                 # 正解：その次の1文字
    return x, y

g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 4, g)
print(x.shape, y.shape)
for i in range(4):
    print(repr("".join(itos[j] for j in x[i].tolist())), "→", repr(itos[y[i].item()]))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([4, 8]) torch.Size([4])
'私の注意を母より' → 'は'
'にしてやりたかっ' → 'た'
'き移ってからも、' → '当'
'を見ていた。私は' → '私'
```

</details>

（どの窓が選ばれるかは乱数で決まります。）`x` の形は `[4, 8]`。4問ぶんの、8文字ずつの入力です。

#### 📝 確認問題（`ch09-q01` · Lv.2 · 長さ T の文脈窓 data[i:i+T] と次の1文字 data[i+T] の対応）

長さ `T = 3` の窓で入力と正解の1文字を取り出すスライスを確認します。次のコードを実行すると何が表示されますか？

```python
data = [10, 20, 30, 40, 50]
T = 3
i = 1
x = data[i:i + T]
y = data[i + T]
print(x, y)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Python のスライス `data[i:i + T]` は開始位置 `i` を含み、終了位置 `i + T` は含まないルールを確認しましょう。
- **ヒント 2**：`i = 1, T = 3` なので `x = data[1:4]`（インデックス 1, 2, 3）、`y = data[4]` になります。それぞれの値を並べてみましょう。
- **期待される出力**：`[20, 30, 40] 50`

**考え方**：Python のスライス `data[i:i+T]` は右端 `i+T` を含まないため、ちょうど `data[i+T]` が窓の直後の「次の1文字（正解）」になります！

1. `i = 1, T = 3` → `i + T = 4`。
1. `data[1:4]` → インデックス 1, 2, 3 の `[20, 30, 40]`。
1. `data[4]` → `50`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q01

</details>

## 埋め込み：文字を「点」にする

ニューラルネットワークに文字を渡すには、文字を数にしなければなりません。第6章のone-hot（2061個の数のうち1つだけ1）は、無駄が多いうえに、どの文字どうしも「同じくらい違う」ものとして扱ってしまいます。

そこで、各文字に **少数の数の組** を割り当てます。たとえば2個の数なら、各文字は平面上の1つの点になります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
torch.manual_seed(0)
C = torch.randn(V, 2)          # 2061文字 × 2個の数（最初はでたらめ）
print("「先」の座標：", C[stoi["先"]])
print("「奥」の座標：", C[stoi["奥"]])
print(C[x].shape)              # 4問 × 8文字 × 2個の数

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
「先」の座標： tensor([-0.4445,  0.8185])
「奥」の座標： tensor([0.8115, 0.1629])
torch.Size([4, 8, 2])
```

</details>

> **💡 理論のポイント：埋め込み（embedding）**
>
> 文字（やトークン）ごとに割り当てた数の組（ベクトル）を **埋め込み**（embedding）と言います。表 `C` の各行が、その文字の埋め込みです。
>
> ポイントは、**この座標も学習で決める**こと。最初はでたらめな位置ですが、学習が進むと「同じような場面で使われる文字」が近くに集まってくるように動いていきます。「先」と「奥」が近くに来れば、「先生と」を見て学んだことが「奥さんと」にも少し役立つようになる。数えるモデルにはできなかった **汎化**（はんか、generalization：見たことのないものへの応用）が起きるのです。
>
> 実際のモデルでは、2個ではなく数十〜数千個の数を使います（この章では32個、Qwen2.5-0.5Bでは896個）。たくさんの数を使えば、「品詞っぽさ」「漢字っぽさ」「人っぽさ」……のように、たくさんの特徴を同時に表せます。

`C[x]` の形 `[4, 8, 2]` は、「4問 × 8文字 × 各文字2個の数」です。第6章で、one-hot × 表 ＝ 表の行の取り出し、でしたね。`C[x]` は、それを番号で直接やっています。

<details><summary><b>基礎の補足：埋め込みルックアップ `C[x]` vs one-hot 行列積</b>（クリックで開く）</summary>

| 比較項目 | `F.one_hot(x, V).float() @ C` | `C[x]`（＝ `nn.Embedding(V, d)(x)`） |
|---|---|---|
| 計算結果 | `C` の `x` 行目を取り出す | **数学的に完全に同じ！** |
| 計算量・メモリ | 2060個の `0` にも掛け算して巨大行列を作る | 行を直接引くだけ（圧倒的に高速・省メモリ） |
| 形の変化 | `[B, T]` → `[B, T, V]` → `[B, T, d]` | `[B, T]` → `[B, T, d]` |

`x` の1マス（文字ID）が、そのまま長さ `d` のベクトル（座標）に置き換わるので、形は常に `[B, T] → [B, T, d]` になります。

</details>

#### 📝 確認問題（`ch09-q02` · Lv.2 · 埋め込み C[x] と one-hot × C の関係、および汎化が生まれる理由）

埋め込み表 `C`（形 `[V, d]`）に対して `C[x]`（または `nn.Embedding(V, d)(x)`）を行うことと、one-hot 表現を直接使うことの違いとして**正しい**のはどれですか？

- **a.** 埋め込み表現を使っても文字 ID が隣り合う文字（例：ID 10 と ID 11）同士しか近くになれず、ID が離れた似た役割の文字は表現できない
- **b.** 埋め込み表 `C` の各文字の座標は人間が国語辞典や品詞情報をもとに手作業で設定する固定値であり、学習ループ中は一切更新されない
- **c.** `C[x]` で得られる密な埋め込みベクトルは one-hot ベクトルより次元数が必ず大きくなり、語彙数 $V$ 以上の膨大なメモリを消費する
- **d.** `C[x]` は `one_hot(x) @ C` と同じ計算を高速に行い、各文字を学習可能な密な座標へ写して「似た文字が近くに集まる＝汎化」を可能にする

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：one-hot 表現ではすべての文字ペアが直交して無関係扱いになりますが、埋め込み空間では「似た文脈に出る文字」がどう配置されるでしょう？
- **ヒント 2**：`C[x]` は `one_hot(x) @ C` と数学的に同一の行取り出しを高速に行い、学習で似た文字を近い座標へ集めることで未知の並びにも汎化できます。
- **正解**：`d`

**考え方**：one-hot ではどの2文字も完全に直交（同じくらい無関係）ですが、埋め込み空間では似た文字が近い点に配置されるため、見たことのない文字の並びにも応用（汎化）が効きます。

1. 計算面：`C[x]` ＝ `one_hot(x) @ C`（無駄な0の掛け算を省略した行ルックアップ）。
1. 表現面：2061次元のスカスカな one-hot を、特徴が詰まった 32次元の座標へ圧縮。
1. 汎化面：似た役割の文字が近い座標に集まるため、未知の8文字並びにも推測が効く。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q02

</details>

#### 📝 確認問題（`ch09-q03` · Lv.2 · 埋め込み表 nn.Embedding(V, d) のパラメータ数の計算）

語彙が $V = 5000$ 文字、埋め込みの大きさ（次元数）が $d = 64$ のとき、埋め込みの表 `nn.Embedding(5000, 64)` のパラメータ数はいくつですか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`nn.Embedding(V, d)` はバイアスを持たない `[V, d]` の表そのものです。
- **ヒント 2**：行数 `V = 5000` と列数 `d = 64` の掛け算でパラメータ総数を求めましょう。
- **正解**：`320000`

**考え方**：埋め込み層にはバイアスがなく、`V × d` の行列そのものがパラメータです。

1. 埋め込み表の形：`[V, d] = [5000, 64]`。
1. パラメータ数：$5000 \times 64 = 320,000$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q03

</details>

## 層を重ねる：多層パーセプトロン

8文字ぶんの埋め込みが手に入ったら、それを1列に並べて、計算を2段重ねます。

1. **隠れ層**：埋め込みを並べたもの（8文字 × 32 = 256個の数）を、256個の「ニューロン」で変換する
2. **出力層**：その結果から、2061文字それぞれの点数（ロジット）を出す

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
d, H = 32, 256    # 埋め込みの大きさ、隠れ層の大きさ
torch.manual_seed(0)
C = torch.randn(V, d)
W1 = torch.randn(T * d, H) * 0.1
b1 = torch.zeros(H)
W2 = torch.randn(H, V) * 0.1
b2 = torch.zeros(V)

emb = C[x]                       # [4, 8, 32]
flat = emb.view(4, T * d)        # [4, 256]：8文字ぶんを横一列に並べる
h = torch.tanh(flat @ W1 + b1)   # [4, 256]：隠れ層
logits = h @ W2 + b2             # [4, 2061]：次の文字の点数
print(emb.shape, flat.shape, h.shape, logits.shape)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
torch.Size([4, 8, 32]) torch.Size([4, 256]) torch.Size([4, 256]) torch.Size([4, 2061])
```

</details>

> **🐍 Python・PyTorchのポイント：.view と .reshape で形を変える**
>
> `emb.view(4, T * d)` は、中身の数はそのままで、形だけを `[4, 8, 32]` から `[4, 256]` に変えます。8文字ぶんの32個ずつを、横一列（256個）に並べ直すイメージです。`-1` を使うと「残りは自動で計算して」という意味になり、`emb.view(4, -1)` でも同じ結果になります。なお、`transpose` 後などでメモリが不連続なテンソルでは `.view()` がエラーになるため、安全な `.reshape(4, -1)`（第6章・[付録B](https://sakunokyumei.github.io/llm-research-textbook/b-python.html)）を使うこともよくあります。

<details><summary><b>基礎の補足：`.view(B, -1)` の形の算数と、MLP全体の shape パイプライン</b>（クリックで開く）</summary>

`.view()` は**要素の総数を保ったまま**形だけを並べ替える操作です。`emb` の形が `[B, T, d] = [4, 8, 32]` のとき：
- 総要素数は $4 \times 8 \times 32 = 1024$ 個
- `emb.view(4, -1)` は「第0軸を `4` に固定し、残りの第1軸は自動で割ってね」の意味 → $1024 \div 4 = 256$（ちょうど $T \times d = 8 \times 32$）！
- コードで `e.view(len(x), -1)` と書けば、バッチサイズ `B` が訓練時(`256`)・検証時(`16239`)・生成時(`1`)で変わっても同じ1行で動きます。

MLP全体のテンソル形の流れを1本の地図で押さえておきましょう：

| ステップ | 計算 | テンソルの形（`shape`） | 意味 |
|---|---|---|---|
| 入力 | `x` | `[B, T]` = `[4, 8]` | $B$ 問 × 直前 $T$ 文字のID |
| 埋め込み | `e = C[x]` | `[B, T, d]` = `[4, 8, 32]` | 各文字を $d$ 次元の座標に変換 |
| 平坦化 | `flat = e.view(B, -1)` | `[B, T*d]` = `[4, 256]` | 8文字ぶんを横1列に連結 |
| 隠れ層 | `h = tanh(flat @ W1 + b1)` | `[B, H]` = `[4, 256]` | $H$ 個のニューロンで特徴抽出 |
| 出力層 | `logits = h @ W2 + b2` | `[B, V]` = `[4, 2061]` | 全語彙 $V$ 文字の生の点数 |

</details>

> **💡 理論のポイント：ニューロンと全結合層**
>
> `flat @ W1 + b1` が、ニューラルネットワークの基本部品 **全結合層**（linear layer）です。数式では
>
> $$\boldsymbol{y} = \boldsymbol{x} W + \boldsymbol{b}$$
>
> と書きます。出力の1つ1つ（ニューロン）は、「入力の数それぞれに重みを掛けて全部足し、最後に $b$（バイアス）を足したもの」です。
>
> <div class="math-display">ニューロン ＝ 入力₁×重み₁ ＋ 入力₂×重み₂ ＋ … ＋ 入力₂₅₆×重み₂₅₆ ＋ バイアス</div>
>
> 1つのニューロンは「どの特徴をどれくらい重視するか」の投票係のようなものです。256個のニューロンが、それぞれ違う観点で8文字を眺めて、点数をつけます。行列の掛け算 `@` は、この256個のニューロンの計算を一度にやってしまう技でした（第6章）。
>
> この「全結合層を何段か重ねたもの」を **多層パーセプトロン**（multilayer perceptron、**MLP**）と呼びます。

#### 📝 確認問題（`ch09-q04` · Lv.2 · .view(B, -1) の形変換と要素数保存の計算）

テンソル `emb` の形が `[8, 6, 16]`（8問 × 6文字 × 16次元）のとき、`flat = emb.view(8, -1)` を実行すると `flat` の形は `[8, ?]` になります。`?` に入る整数はいくつですか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`.view()` はテンソルの総要素数を変えずに形だけを並べ替えます。第0軸を `8` に固定したとき、残りの2軸はどうまとまるでしょう？
- **ヒント 2**：`[B, T, d]` に `.view(B, -1)` を掛けると第1軸は `T × d` になります。ここでは `6 × 16` を計算しましょう。
- **正解**：`96`

**考え方**：`.view()` は総要素数を変えずに形だけを並べ替えます。`[B, T, d]` → `.view(B, -1)` で `[B, T * d]` になるため、`[8, 6, 16]` は `[8, 96]` になります。

1. 元の総要素数：$8 \times 6 \times 16 = 768$。
1. 第0軸を `8` に固定したので、第1軸は $768 \div 8 = 96$（$= 6 \times 16$）。
1. よって `flat.shape` は `[8, 96]`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q04

</details>

#### 📝 確認問題（`ch09-q05` · Lv.2 · MLP の形パイプライン [B,T]→[B,T,d]→[B,Td]→[B,H]→[B,V] とエラー診断）

窓の大きさ `T = 8`, 埋め込み `d = 32`, 隠れ層 `H = 256`, 語彙 `V = 2061` の MLP にバッチサイズ `B = 4` の入力 `x` を通したとき、テンソルの形の変化として**正しい**流れはどれですか？

- **a.** `[4, 8]` →（出力層）`[4, 2061]` →（隠れ層+tanh）`[4, 256]` →（view）`[4, 8, 32]` →（埋め込み）`[4, 32]`
- **b.** `[4, 8]` →（埋め込み）`[4, 8, 32]` →（view）`[4, 256]` →（隠れ層+tanh）`[4, 256]` →（出力層）`[4, 2061]`
- **c.** `[4, 8]` →（埋め込み）`[4, 8, 32]` →（view）`[32, 32]` →（隠れ層+tanh）`[32, 256]` →（出力層）`[4, 2061]`
- **d.** `[4, 8]` →（埋め込み）`[4, 8, 256]` →（view）`[4, 256]` →（隠れ層+tanh）`[4, 256]` →（出力層）`[4, 1]`

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：先頭のバッチサイズ `B = 4` は最初から最後まで不変で、最終出力の最後の次元は「次の文字の候補数（語彙数 `V`）」になります。
- **ヒント 2**：順番に「入力 `[4, 8]` → 埋め込み `[4, 8, d]` → 平坦化 `[4, T*d]` → 隠れ層 `[4, H]` → 出力 `[4, V]`」と当てはめましょう。
- **正解**：`b`

**考え方**：ニューラルネットワークのデバッグの9割は「各層の入出力の `shape` を追うこと」で解決できます。

1. 入力 `x`：`[4, 8]`（文字ID）。
1. `self.emb(x)`：`[4, 8, 32]`。
1. `.view(4, -1)`：`[4, 256]`（$= 8 \times 32$）。
1. `tanh(self.fc1(...))`：`[4, 256]`（隠れ層 `H`）。
1. `self.fc2(...)`：`[4, 2061]`（語彙 `V` ぶんのロジット）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q05

</details>

*(この節の追加問題 `ch09-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html で自動採点・復習できます)*

## なぜ tanh を挟むのか：非線形性

隠れ層の計算で、`torch.tanh(…)` という関数を通しています。これを **活性化関数**（activation function）と言います。どんな形の関数か、見てみましょう。

📦 **準備セル**：ライブラリや教材ファイルを取得します。エラーが出た場合は `-q` を外して詳細を確認してください。

In [ ]:
!pip install -q matplotlib-fontja

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja

z = torch.linspace(-4, 4, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(z, torch.tanh(z), label="tanh")
plt.plot(z, torch.relu(z), label="ReLU")
plt.plot(z, F.gelu(z), label="GELU", linestyle="--")
plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.ylim(-1.5, 3)
plt.legend()
plt.title("いろいろな活性化関数")
plt.show()

![tanh・ReLU・GELUの3つの活性化関数のグラフ。どれも直線ではなく、折れ曲がったりS字になったりしている](https://sakunokyumei.github.io/llm-research-textbook/img/figs/09-mlp/cell5_0.png)

#### 📐 数学のポイント：なぜ非線形（`tanh`）が必須か：2×2の具体数で「線形崩壊」を証明する

活性化関数がないと、全結合層を何段重ねても、結局は**1段の全結合層**に潰れてしまいます。

$$(xW_1)W_2 = x(W_1 W_2) = xW'$$

**具体的な2×2の数字で確かめてみましょう！** 入力 $x = [1, 2]$、第1層 $W_1 = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}$、第2層 $W_2 = \begin{pmatrix} 2 & -1 \\ 1 & 3 \end{pmatrix}$ とします。

- **活性化なし（2段を順番に計算）**：
  1. 第1層：$x W_1 = [1\cdot 1 + 2\cdot 0, \; 1\cdot 2 + 2\cdot 1] = [1, 4]$
  2. 第2層：$(x W_1) W_2 = [1\cdot 2 + 4\cdot 1, \; 1(-1) + 4\cdot 3] = [6, 11]$
- **1つの行列 $W' = W_1 W_2$ に合体して1段で計算**：
  - $W' = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}\begin{pmatrix} 2 & -1 \\ 1 & 3 \end{pmatrix} = \begin{pmatrix} 4 & 5 \\ 1 & 3 \end{pmatrix}$
  - $x W' = [1\cdot 4 + 2\cdot 1, \; 1\cdot 5 + 2\cdot 3] = [6, 11]$（**どんな $x$ を入れても完全に一致！**）

```python no-run
import math
# 活性化なしでは (x @ W1) @ W2 == x @ (W1 @ W2) で1層に潰れる
# 一方 tanh を間に挟むと…
h = [math.tanh(1), math.tanh(4)]   # [0.7616, 0.9993]（まっすぐの比率が曲がる！）
y = [h[0]*2 + h[1]*1, h[0]*(-1) + h[1]*3]  # [2.5225, 2.2364] → どんな1枚の行列 W' でも再現不可！
```

たとえば「数を2倍して、さらに3倍する」操作は、最初から「6倍する（$2 \times 3 = 6$）」操作1回と同じですよね。掛け算だけなら、何十層重ねようが結局1段に潰れてしまいます。

そこで、層と層の間に「**直線ではない**（非線形な）」関数を挟みます。折れ曲がったり、S字になったりする関数を途中に挟むことで、1回にまとめることができなくなり、複雑で曲がりくねった現実の言葉のパターンを自在に表現できるようになります。

- **tanh**（ハイパボリックタンジェント）：-1から1の間のS字。この章で使う、昔ながらの関数
- **ReLU**（レルー）：マイナスは0、プラスはそのまま。単純で速いので大人気
- **GELU**（ジェルー）：ReLUをなめらかにしたもの。GPTで使われている（第12章）

#### 📝 確認問題（`ch09-q07` · Lv.2 · 活性化関数（tanh等）を抜くと2層が1層に潰れる理由）

MLP の隠れ層から活性化関数 `torch.tanh` を取り除いて `h = self.fc1(flat)` にしてしまうと、モデルの表現力はどうなりますか？

- **a.** 活性化関数が消えたことで `W1` と `W2` の重みテンソルが自動的に統合され、モデルのパラメータ数がちょうど半分に減ってしまう
- **b.** $(x W_1) W_2 = x (W_1 W_2) = x W'$ となり、何層重ねても結局 1 段の全結合層（線形変換）と同じ表現力に潰れてしまう
- **c.** 途中の曲がりがなくなって計算が単純になるため、むしろ複雑な言語パターンを 2 倍の精度で表現できるようになる
- **d.** PyTorch の計算グラフは線形層だけを連続して接続すると微分できなくなるため、`loss.backward()` で必ず実行時エラーが起きる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「2倍してから3倍する」操作は「最初から6倍する」1回と同じです。行列の掛け算を2回連続で行うと結合法則でどう書けるでしょう？
- **ヒント 2**：$(x W_1) W_2 = x (W_1 W_2)$ と1つの行列 $W' = W_1 W_2$ にまとまってしまいます。バイアスがあっても同様に1段の線形変換に退化します。
- **正解**：`b`

**考え方**：非線形な活性化関数（`tanh`, `ReLU`, `GELU`）を間に挟むことで初めて「1つの行列にまとめられない複雑な曲がり」を表現でき、層を深くする意味が生まれます。

1. 活性化なし：$(x W_1) W_2 = x (W_1 W_2)$。$W' = W_1 W_2$ と置けば1層と同じ。
1. バイアスがあっても $(x W_1 + b_1) W_2 + b_2 = x(W_1 W_2) + (b_1 W_2 + b_2)$ でやはり1層に潰れる。
1. `tanh` などの非線形関数を挟むと結合法則でまとめられなくなり、深い層が本領を発揮する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q07

</details>

#### 📝 確認問題（`ch09-q08` · Lv.4 · 【自力実装】2層線形層の崩壊 (xW1)W2 = x(W1W2) と tanh 非線形性の検証）

ベクトル `x` と行列 `W` の積を計算する `vec_mat(x, W)` と、2×2行列の積 `mat_mat(A, B)` を使って、活性化なしでは `(x @ W1) @ W2` と `x @ (W1 @ W2)` が完全に一致すること、一方 `tanh` を挟むと異なる結果になることを純Pythonで実装・検証してください。

**ひな形コード：**
```python
import math

def vec_mat(x, W):
    # x (長さ m) と W (m × n) の積（長さ n のリスト）を返す
    pass

def mat_mat(A, B):
    # 行列 A (m × k) と B (k × n) の積 (m × n) を返す
    pass

def two_layer_tanh(x, W1, W2):
    # h = [math.tanh(v) for v in vec_mat(x, W1)] のあと vec_mat(h, W2) を返す
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`vec_mat(x, W)` は各列 `j` ごとに `x` と `W` の `j` 列目の内積を求めます。`mat_mat(A, B)` は `A` の各行に `vec_mat` を適用すれば書けます。
- **ヒント 2**：`vec_mat` の `j` 番目は `sum(x[i] * W[i][j] for i in range(len(x)))`、`two_layer_tanh` は `vec_mat(x, W1)` の各要素に `math.tanh` を通してから `vec_mat(..., W2)` を返します。

**解答例：**
```python
import math

def vec_mat(x, W):
    n = len(W[0])
    return [sum(x[i] * W[i][j] for i in range(len(x))) for j in range(n)]

def mat_mat(A, B):
    return [vec_mat(row, B) for row in A]

def two_layer_tanh(x, W1, W2):
    h = [math.tanh(v) for v in vec_mat(x, W1)]
    return vec_mat(h, W2)
```

**検算テスト（Colabで実行して確認）：**
```python
x = [1.0, 2.0]
W1 = [[1.0, 2.0], [0.0, 1.0]]
W2 = [[2.0, -1.0], [1.0, 3.0]]
# 1. 活性化なし：(x @ W1) @ W2 == x @ (W1 @ W2)
step_by_step = vec_mat(vec_mat(x, W1), W2)
W_merged = mat_mat(W1, W2)
single_layer = vec_mat(x, W_merged)
assert step_by_step == [6.0, 11.0]
assert single_layer == [6.0, 11.0]
assert all(abs(a - b) < 1e-9 for a, b in zip(step_by_step, single_layer))
# 2. tanh あり：入力を2倍しても出力が2倍にならない（非線形！）
y1 = two_layer_tanh([1.0, 2.0], W1, W2)
y2 = two_layer_tanh([2.0, 4.0], W1, W2)
assert abs(y2[0] - 2.0 * y1[0]) > 0.5
print("ALL PASSED")
```

**考え方**：線形変換は $f(2x) = 2f(x)$ を必ず満たしますが、`tanh` を挟むと $f(2x) \neq 2f(x)$ となり、1つの行列では決して真似できない非線形な関数が生まれます。

1. `vec_mat(x, W)`：各列 `j` について内積 `sum(x[i] * W[i][j])` を計算。
1. `mat_mat(A, B)`：`A` の各行ベクトルに `vec_mat(row, B)` を適用。
1. `two_layer_tanh`：第1層の出力に `math.tanh` を通してから第2層 `W2` を掛ける。
1. `[1, 2]` と `[2, 4]` で「出力が2倍にならない＝非線形」を数値で確認。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q08

</details>

## 部品をまとめる：クラスで書く

ここまでの部品（`C`、`W1`、`b1`、`W2`、`b2`）をバラバラに持っていると、管理が大変です。PyTorchでは、モデルを **クラス** という設計図にまとめて書きます。

PyTorchの `nn.Module` に入る前に、まずは**ふつうのPythonだけの最小コード**で `class`・`__init__`・`self`・継承・`super().__init__()` をスッキリ理解しておきましょう！

<details><summary><b>基礎の補足：理解のための最小コード：PyTorchの前に「ふつうのPythonクラス」を押さえる</b>（クリックで開く）</summary>

**① 基本のクラス（設計図）と `__init__`・`self`**：
関数を「部品（データ）と処理のセット」にまとめたものが **クラス** です。

```python no-run
class LinearRule:                  # 「LinearRule」という名前の設計図
    def __init__(self, a, b):      # 実物を作るときに1回だけ呼ばれる初期化関数
        self.a = a                 # self（＝作られた実物自身）に部品 a を保存
        self.b = b                 # 部品 b を保存

    def calc(self, x):             # 実物が持つメソッド（関数）
        return self.a * x + self.b # 保存しておいた self.a, self.b を使って計算

rule = LinearRule(3, 1)            # 設計図から実物（インスタンス）を1個作る
print(rule.calc(2))                # → 3 * 2 + 1 = 7
```

**② 継承 `class 子(親):` と `super().__init__()`**：
すでにある便利な設計図（親クラス）の機能を丸ごと引き継いで、自分用の部品を足す仕組みを **継承** と呼びます。

```python no-run
class BaseTool:
    def __init__(self):
        self.registry = []         # 親が用意する管理台帳

class MyTool(BaseTool):            # BaseTool の機能をそのまま引き継ぐ！
    def __init__(self, scale):
        super().__init__()         # 親（BaseTool）の __init__ を必ず先に実行して台帳を作る
        self.scale = scale

    def forward(self, x):
        return x * self.scale
```

**③ PyTorch の `class MLP(nn.Module):` との1対1対応**：
- `class MLP(nn.Module):` → PyTorchの親クラス `nn.Module` を継承（すると `model.parameters()` や `.to("cuda")` や `state_dict()` が全部自動で使えるようになる！）
- `super().__init__()` → 親 `nn.Module` のパラメータ管理台帳を初期化（忘れるとエラー！）
- `self.emb = ...` → 必ず `self.` を付けて代入することで、台帳に部品が自動登録される
- `def forward(self, x):` → 順伝播の計算。使うときは `model.forward(x)` ではなく **`model(x)`** と呼ぶのが作法

</details>

それでは、MLPを `nn.Module` のクラスにまとめてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
class MLP(nn.Module):
    def __init__(self, V, T, d=32, H=256):
        super().__init__()
        self.emb = nn.Embedding(V, d)     # 埋め込みの表（C）
        self.fc1 = nn.Linear(T * d, H)    # 隠れ層（W1, b1）
        self.fc2 = nn.Linear(H, V)        # 出力層（W2, b2）

    def forward(self, x):
        e = self.emb(x)                          # [B, T, d]
        h = torch.tanh(self.fc1(e.view(len(x), -1)))  # [B, H]
        return self.fc2(h)                       # [B, V]

torch.manual_seed(0)
model = MLP(V, T)
print(model)
print("パラメータの数：", sum(p.numel() for p in model.parameters()))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
MLP(
  (emb): Embedding(2061, 32)
  (fc1): Linear(in_features=256, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=2061, bias=True)
)
パラメータの数： 661421
```

</details>

> **🐍 Python・PyTorchのポイント：クラスは「設計図」**
>
> `class MLP(nn.Module):` は「`MLP` という名前の設計図を作る。`nn.Module`（PyTorchのモデルの基本形）を土台にする」という意味です。
>
> - `__init__`：設計図から実物を作るとき（`MLP(V, T)`）に最初に実行される準備。ここで部品を用意する
> - `self`：作られた実物自身のこと。`self.emb` は「この実物が持っている `emb`」
> - `super().__init__()`：土台（`nn.Module`）の準備も忘れずにやる、というお約束
> - `forward`：入力を受け取って出力を返す計算。`model(x)` と書くと、これが呼ばれる
>
> CSSのクラスを思い出してください。`.button { … }` という1つの定義から、ページ中のたくさんのボタンに同じ見た目が適用されますね。Pythonのクラスも、1つの設計図から同じ仕組みの実物（**インスタンス**）をいくつでも作れます。

#### 💡 理論のポイント：パラメータの数え方

パラメータの数を手で確かめてみましょう。全結合層は「入力の数 × 出力の数」個の重みと、「出力の数」個のバイアスを持っています。

| 部品 | 計算 | 数 |
|---|---|---|
| 埋め込み | 2061 × 32 | 65,952 |
| 隠れ層の重み＋バイアス | 256 × 256 ＋ 256 | 65,792 |
| 出力層の重み＋バイアス | 256 × 2061 ＋ 2061 | 529,677 |
| 合計 | | 661,421 |

`print` の結果と一致しましたか？ パラメータの8割が**出力層**に集中していますね。語彙（2061文字）のそれぞれに点数を出すための重みが必要だからです。本物のLLMでも、語彙が15万もあると、出力層と埋め込みだけで大きな割合を占めます（第1章のQwenでは、約4.9億個のうち約1.4億個が埋め込み）。

#### 📝 確認問題（`ch09-q09` · Lv.2 · Python の class・__init__・self・継承 super().__init__() の動作）

PyTorch の `nn.Module` の土台になる Python の `class`・`__init__`・`self` の動きを確認します。次のコードを実行すると何が表示されますか？

```python
class Counter:
    def __init__(self, start):
        self.value = start

    def add(self, n):
        self.value += n
        return self.value

c = Counter(10)
print(c.add(3), c.add(5))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：インスタンス `c` の中の `self.value` は、メソッド `c.add(...)` を呼ぶたびに上書きされて保持され続けます。
- **ヒント 2**：最初は `c.value = 10` です。1回目の `c.add(3)` でいくつになり、そこからさらに `c.add(5)` を足すといくつになるか順に追いましょう。
- **期待される出力**：`13 18`

**考え方**：`self` は作られた実物（インスタンス `c`）自身を指す変数で、`self.value` に状態（部品やデータ）を保持し続けられます。

1. `c = Counter(10)` → `__init__(c, 10)` が呼ばれ `c.value = 10`。
1. `c.add(3)` → `c.value` が `13` になり `13` を返す。
1. `c.add(5)` → `c.value` が `18` になり `18` を返す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q09

</details>

#### 📝 確認問題（`ch09-q10` · Lv.2 · PyTorch nn.Module のお約束（super().__init__・self登録・model(x)））

PyTorch で `class MLP(nn.Module):` を定義するときのルールとして、**正しい**説明はどれですか？

- **a.** `__init__` の中で `self.` を付けず `fc1 = nn.Linear(...)` とローカル変数に置いても、PyTorch が関数内の変数を走査して `model.parameters()` へ自動登録してくれる
- **b.** 順伝播を実行するときは毎回 `model.__init__(x)` を直接呼び出し、初期化メソッドの戻り値として次の文字のロジットテンソルを受け取る決まりになっている
- **c.** `__init__` 先頭で `super().__init__()` を呼び、層を `self.fc1 = ...` に代入することで `model.parameters()` が重みを収集でき、`model(x)` で `forward` が呼ばれる
- **d.** `super().__init__()` は読みやすさのための慣習的な飾りにすぎないため、省略しても `self.emb = nn.Embedding(...)` の代入や誤差逆伝播の学習は問題なく動く

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`nn.Module` を継承したクラスが、中にあるすべての層の重みを `model.parameters()` で自動収集するために必要な 2 つのお約束を思い出しましょう。
- **ヒント 2**：`super().__init__()` で親クラスの部品台帳を初期化し、`self.層名 = ...` で台帳へ部品を登録します。呼び出し時は `model(x)` が `forward` を実行します。
- **正解**：`c`

**考え方**：`nn.Module` を継承する最大の恩恵は、`model.parameters()`・`model.to(device)`・`model.state_dict()` といった便利機能が自動で動くようになることです。

1. `super().__init__()`：`nn.Module` 内部の部品台帳を初期化する。
1. `self.emb = nn.Embedding(...)`：台帳に部品を自動登録する。
1. `model(x)`：`nn.Module` の仕組み経由で `forward(self, x)` を呼び出す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q10

</details>

*(この節の追加問題 `ch09-q11` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html で自動採点・復習できます)*

## 学習させる

学習ループは第8章とほぼ同じです。今回は、パラメータの更新を **オプティマイザ**（optimizer）という道具に任せます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def val_loss(model):
    X = torch.stack([val_data[i:i + T] for i in range(len(val_data) - T)])
    Y = val_data[T:]
    return F.cross_entropy(model(X), Y).item()

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = MLP(V, T)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for step in range(10001):
    x, y = get_batch(train_data, 256, g)
    loss = F.cross_entropy(model(x), y)
    optimizer.zero_grad()     # 勾配を0に
    loss.backward()           # 勾配を計算
    optimizer.step()          # パラメータを更新
    if step % 2000 == 0:
        print(f"ステップ {step:5d}：訓練 {loss.item():.3f}  検証 {val_loss(model):.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ     0：訓練 7.659  検証 7.662
ステップ  2000：訓練 3.594  検証 3.919
ステップ  4000：訓練 3.591  検証 3.677
ステップ  6000：訓練 3.178  検証 3.532
ステップ  8000：訓練 3.333  検証 3.448
ステップ 10000：訓練 2.734  検証 3.395
```

</details>

CPUで2分ほどかかります。コーヒーでも淹れてきてください。……戻ってきましたか？ 検証損失が **3.395**。数えるbigramの3.581を、ついに破りました！

> **🐍 Python・PyTorchのポイント：オプティマイザ**
>
> `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` の3行は、第7章で手で書いた「勾配を0に → 勾配を計算 → `w -= lr * w.grad`」と同じことをしています。`torch.optim.SGD` は、モデルの全パラメータについて、まとめて `w -= lr * w.grad` をやってくれる係です。<br>
> `@torch.no_grad()` を関数の上に付けると、その関数の中身全体が `with torch.no_grad():` の中にあるのと同じになります（関数の上に置く `@` は **デコレータ** という印で、第6章の行列の掛け算の `@` とは別物です）。

#### 📝 確認問題（`ch09-q12` · Lv.2 · torch.optim.SGD の3行と @torch.no_grad() デコレータの意味）

`optimizer = torch.optim.SGD(model.parameters(), lr=0.1)` を使った3行 `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()` は、第7章の手書きループの何に対応していますか？

- **a.** `zero_grad()`＝全パラメータの勾配リセット、`backward()`＝勾配計算、`step()`＝`with torch.no_grad():` 内で全パラメータを `p -= lr * p.grad` で更新
- **b.** `zero_grad()`＝全パラメータの重みを 0 に初期化、`backward()`＝検証データの損失計算、`step()`＝学習済みモデルで文章を 1 文字だけ生成
- **c.** `zero_grad()`＝勾配のリセット、`step()`＝内部で自動的に `loss.backward()` と重み更新の両方を実行するため、実は `loss.backward()` の行は削除してよい
- **d.** `zero_grad()`＝ミニバッチの抽出、`backward()`＝誤差逆伝播を使わずランダム探索で候補生成、`step()`＝一番損失の低い重みを採用して上書き

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第7〜8章ではパラメータが1個だったので手で勾配リセットと `W -= lr * W.grad` を書けましたが、MLP では部品が5個以上に増えました。
- **ヒント 2**：オプティマイザは新しい計算ではなく、`zero_grad()` で全パラメータの `.grad` を消し、`step()` で `no_grad` 内の更新をまとめて代行しています。
- **正解**：`a`

**考え方**：オプティマイザは新しい魔法ではなく、第7章で手で書いた「勾配リセット」と「`no_grad` 内での更新」を全パラメータぶん自動化してくれる道具です。

1. `optimizer.zero_grad()`：登録された全パラメータの `.grad` をリセット。
1. `loss.backward()`：計算グラフを逆にたどり全パラメータの `.grad` を計算。
1. `optimizer.step()`：全パラメータに `p -= lr * p.grad` を実行。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q12

</details>

## 記録表を更新

| モデル | 章 | 検証データの損失 |
|---|---|---|
| でたらめ | 5 | 7.631 |
| bigram（数える） | 5 | 3.581 |
| ニューラルbigram | 8 | 3.685 |
| **MLP（直前8文字、SGD 1万ステップ）** | **9** | **3.395** |

## 書かせてみる

窓を1文字ずつずらしながら生成します。最初の8文字は自分で与えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
@torch.no_grad()
def generate(model, start, n=100, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = [stoi[c] for c in start]
    for _ in range(n):
        x = torch.tensor([ids[-T:]])               # 直前8文字
        probs = F.softmax(model(x), dim=1)[0]
        ids.append(torch.multinomial(probs, 1, generator=g).item())
    return "".join(itos[i] for i in ids)

print(generate(model, "私はその人を常に"))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
私はその人を常によう加打で返事が物った。先生と奥さんはたしようで。ただの光をある時がら見た。
「今は他いに私を煎にかず話する果べ橋当に告配夫明問にする親ないでも事去上かに故このよに冷座来なく遥えますか」
　奥さんまし
```

</details>

「返事が」「先生と奥さんは」「ただの光を」。bigramよりも、言葉のまとまりがずっと自然になりました。かぎかっこ「 」も閉じていますが、これは**たまたま**です。開いたかぎかっこから閉じるまで50文字ほどあり、直前8文字しか見えないこのモデルは、かぎかっこを開いたことを覚えていられません。遠くまで覚えておく仕組みは、第11章で作ります。……中身は相変わらず支離滅裂ですが、一歩前進です。

#### 📝 確認問題（`ch09-q13` · Lv.2 · 固定長8文字窓 MLP の生成条件と長期記憶の限界）

窓サイズ `T = 8` の MLP で文章を生成するとき、「開いたかぎかっこ `「` を50文字後に正しく閉じる」ことがこのモデルの構造上できない本質的な理由はどれですか？

- **a.** 隠れ層の活性化関数に `tanh` を採用すると出力範囲が -1〜1 に制限され、かぎかっこのような記号文字を出力できなくなるから
- **b.** 入力が毎回 `ids[-8:]`（直前 8 文字）に固定されているため、9 文字以上前にあった `「` の情報がモデルに一切渡らないから
- **c.** 『こころ』から作った語彙表 `stoi` には記号類が含まれず、かぎかっこ `「` や `」` 自体をトークン ID に変換できないから
- **d.** 学習ステップ数が 1 万回と少ないためであり、同じ 8 文字窓のまま 10 万回以上学習させれば 50 文字前の `「` も記憶できるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：生成ループの 1 ステップで、モデルの `forward` に実際に渡されている入力 `x` がどの範囲の文字か確認しましょう。
- **ヒント 2**：`x = torch.tensor([ids[-T:]])` は常に直近 8 文字しか切り出さないため、50 文字前の情報は物理的に入力へ入っていません。
- **正解**：`b`

**考え方**：固定窓 MLP は「8文字より前の出来事を完全に忘れる」構造です。遠くの文字同士を自在に結びつける仕組みが、第11章で学ぶ **Attention** です。

1. 生成の各ステップでモデルが見ているのは `ids[-8:]` の8文字だけ。
1. 50文字前の `「` は窓の外に押し出されており、モデルには見えない。
1. また書き出しが `T` 文字未満だと `ids[-T:]` の長さが `T` に足りず、`fc1` の入力次元（`T * d`）と合わずエラーになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q13

</details>

## 訓練と検証の差に注目

学習の出力をもう一度見てください。最後の訓練損失は2.734、検証損失は3.395。**訓練のほうがずっと良い**ですね。

> **💡 理論のポイント：汎化のギャップ**
>
> 訓練データでは良くて、検証データではそれほどでもない。これは、モデルが訓練データの**クセを覚え始めている**サインです。第5章の「過去問の丸暗記」が始まっているのです。
>
> - 訓練損失と検証損失がどちらも下がっている → 順調
> - 訓練損失だけ下がって、検証損失が下がらない（または上がる） → **過学習**（丸暗記）
>
> 『こころ』は16万字しかないので、66万個のパラメータを持つモデルにとっては、丸暗記できてしまう量なのです。次の章では、この過学習と戦う方法を含め、学習を上手に進める技をまとめて学びます。

<details><summary><b>もっと深く：2003年の元祖ニューラル言語モデル</b>（クリックで開く）</summary>

この章のモデルは、ヨシュア・ベンジオらが2003年に発表した論文「A Neural Probabilistic Language Model」の設計とほぼ同じです。彼らは単語の埋め込みと、このMLPを使って、それまで主流だったn-gramモデルを上回る成績を出しました。当時のコンピュータでは学習に何週間もかかったそうです。あなたはColabで2分。20年の進歩を感じますね。

</details>

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x512 and 256x256)`**：形が合っていません。`T`（窓の大きさ）や `d` を変えたときは、`MLP(V, T)` を作り直しているか確認しましょう。エラーの数字（4x512と256x256）から、どこがずれているか推理できます。
> - **`RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x256 and 512x256)`**（書かせるときに出る）：書き出しの文字数が、窓の大きさ `T` より短いです。書き出しを `T` 文字以上にしましょう。
> - **`TypeError: __init__() missing … argument`**：`MLP(V, T)` のように、必要な引数を渡し忘れています。
> - **`AttributeError: 'MLP' object has no attribute …`**：`self.` を付け忘れて部品を作ると、モデルに部品が登録されません。

> **🔬 やってみよう：窓の大きさを変えてみよう**
>
> 「8文字の窓」の節のセルにある `T = 8` を、`T = 3` や `T = 16` に変えて、そのセルから下を順に実行し直してみましょう（モデルも作り直されます）。`T = 16` にしたら、書かせるときの書き出しも16文字以上にしてください（例：`generate(model, "私はその人を常に先生と呼んでいた。")`）。書き出しが窓より短いと、形が合わないエラーになります。窓が大きいほど検証損失は下がるでしょうか？ 予想してから試してください。（ヒント：大きければいいとは限りません。データが16万字しかないことを思い出して。）

#### 📝 確認問題（`ch09-q14` · Lv.2 · 訓練損失と検証損失のギャップ（過学習の始まり）の読み取り）

第9章のMLPの学習ログでは、ステップ10000で「訓練損失 `2.734`、検証損失 `3.395`」と大きな差がつきました。この現象が示しているサインはどれですか？

- **a.** 語彙数 `V` が 2061 文字と多すぎるため、どんな改良を施しても検証損失は絶対に `3.395` より下がらないという限界の証明
- **b.** 約 66 万パラメータのモデルが 16 万字の訓練データのクセを丸暗記し始めている（**過学習**の始まり）サイン
- **c.** 学習率が実質 `0` に近くて重みが動いておらず、モデルが訓練データから何も学習できていない状態のサイン
- **d.** 検証損失が訓練損失より大きくなるのは評価コードの計算バグであり、本来は必ず検証損失のほうが低くなるはずというサイン

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：練習問題（訓練データ）の成績 `2.734` と、初見の本番（検証データ）の成績 `3.395` の間に大きな開きが生じています。
- **ヒント 2**：データ量（16万字）に対してパラメータ数（66万個）が多いため、訓練データの細部だけを覚え込む「過学習」が始まっています。
- **正解**：`b`

**考え方**：「訓練損失と検証損失の差（汎化ギャップ）」を常に観察することが、過学習を早期発見するための基本です。

1. 訓練・検証がどちらも下がっている間 → 順調に一般的なパターンを学習中。
1. 訓練だけ下がり続け、検証が頭打ち・悪化し始める → 過学習（丸暗記）。
1. 対策（ドロップアウト・重み減衰・データ増量など）を第10章で学ぶ。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q14

</details>

#### 📝 確認問題（`ch09-q15` · Lv.3 · なぜMLPは 2061^8 通りの組み合わせ爆発を越えてbigramに勝てたのか）

直前8文字の並びは $2061^8$ 通り以上あり、検証データに出てくる8文字のほとんどは訓練データに一度も出てきません。それなのに、この章のMLPが数えるbigram（3.581）を破って検証損失 3.395 を達成できた理由を、「埋め込み」と「汎化」という言葉を使って説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「数える n-gram は完全一致しか見られない」のに対し、MLP が文字を「連続なベクトル（座標）」に変換したことで何が起きたか整理しましょう。
- **ヒント 2**：① n-gram は $2061^8$ 通りの組み合わせ爆発で未知の並びに対応できない、②埋め込みで似た役割の文字が近い座標に集まる、③未知の8文字並びにも知識を共有して汎化できる、の3点を結びつけましょう。
- **解答の目安**：数えるn-gramは8文字の完全一致しか使えないため、見たことのない並びでは破綻します。一方MLPは、各文字を学習可能なベクトル（埋め込み）に変換するため、似た文脈で使われる文字同士が近い座標に集まります。その結果、訓練データに一度もなかった8文字の並びでも「似た特徴を持つ並び」として重みを共有して推測できる（汎化が起きる）ため、組み合わせ爆発を起こさず約66万パラメータでbigramを上回る予測ができました。

**考え方**：離散的な「文字の記号」を連続的な「特徴ベクトル（埋め込み）」へ写した瞬間、組み合わせ爆発の呪いが解けて汎化が生まれます。これが2003年Bengio論文から現代LLMまで貫かれる核心アイデアです。

1. n-gramの壁：8文字の表は $2061^8$ 行になり、ほとんどが0回で数えられない。
1. 埋め込みの力：似た役割の文字を近い座標に配置する（パラメータ数はたった `V × d`）。
1. 汎化：未知の8文字並びでも、各位置の特徴ベクトルが似ていれば正しく次文字を予測できる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html#ex-ch09-q15

</details>

**✅ できたらチェック**

- [ ] 8文字の窓で入力と正解を作れた
- [ ] 埋め込みが「文字を数の組で表したもの」で、学習で動くことを説明できる
- [ ] 形 `[B, 8, 32]` → `[B, 256]` → `[B, 256]` → `[B, 2061]` の流れを追えた
- [ ] 活性化関数が必要な理由を説明できる
- [ ] `nn.Module` のクラスでモデルを書き、学習させた
- [ ] 検証損失でbigramの記録を破った

> **📝 まとめ**
>
> - **埋め込み**：文字ごとに学習可能な数の組を割り当てる。似た文字が近づき、**汎化**が生まれる
> - **全結合層** $y = xW + b$ を、**活性化関数**（非線形）を挟んで重ねたものが **MLP**
> - PyTorchでは、モデルを `nn.Module` の **クラス** で書く。`__init__` で部品、`forward` で計算
> - 直前8文字を見るMLPで、検証損失3.395。数えるbigram（3.581）を超えた
> - 訓練と検証の損失の差は、**過学習**の始まりのサイン

**ボンゴ先生**：次の章は「学習のお医者さん」。学習がうまくいかないとき、何を診ればいいか。初期化、正規化、Adam、学習率スケジュール、ドロップアウト……GPTを学習させる前に必要な道具を、一気に救急箱に詰め込もう。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/09-mlp.html
- **次の章**：[第10章 学習のお医者さん：うまく学習させる技](https://sakunokyumei.github.io/llm-research-textbook/10-training-tips.html)